# The logit lens: what each GPT-2 block would predict

The logit lens reads a transformer's residual stream after every block through the model's own final norm and unembedding, showing which next token each depth already favours (nostalgebraist 2020, "interpreting GPT: the logit lens"). Downloads GPT-2 small at a pinned revision (about 550 MB); runs on a laptop CPU in seconds.

In [1]:
# recipe-test: tool=transformers expected=True
import torch
from transformers import AutoTokenizer, GPT2LMHeadModel

import torchlens as tl

REVISION = "607a30d783dfa663caf39e06633721c8d4cfcd7e"  # pragma: allowlist secret
tok = AutoTokenizer.from_pretrained("openai-community/gpt2", revision=REVISION)
model = GPT2LMHeadModel.from_pretrained(
    "openai-community/gpt2", revision=REVISION, use_cache=False
).eval()
prompt = "The capital of France is Paris and the capital of Italy is"
ids = tok(prompt, return_tensors="pt").input_ids
RESULT = model.config.n_layer == 12

.venv-ll/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6522.61it/s]


## With TorchLens

`trace.using("mechinterp").logit_lens()` finds the final norm and unembedding, decodes every block's output through them, and checks that the last block reproduces the model's own logits. `stacked()` returns the per-block logits as one tensor: blocks x batch x positions x vocabulary.

In [2]:
# recipe-test: tool=transformers expected=True
logits = tl.trace(model, ids).using("mechinterp").logit_lens().stacked()
for index, layer_logits in enumerate(logits):
    print(f"block {index}: {tok.decode(layer_logits[0, -1].argmax())!r}")
RESULT = logits.shape[0] == model.config.n_layer

block 0: ' not'
block 1: ' now'
block 2: ' not'
block 3: ' not'
block 4: ' not'
block 5: ' not'
block 6: ' also'
block 7: ' Rome'
block 8: ' Rome'
block 9: ' Rome'
block 10: ' Rome'
block 11: ' Rome'


## The same computation in plain PyTorch

A forward hook on every block collects its output, then each one goes through the final norm and unembedding by hand.

In [3]:
# recipe-test: tool=transformers expected=True
states = []
hooks = [
    block.register_forward_hook(lambda m, i, o: states.append(o[0] if isinstance(o, tuple) else o))
    for block in model.transformer.h
]
with torch.no_grad():
    model(ids)
    plain_logits = torch.stack([model.lm_head(model.transformer.ln_f(state)) for state in states])
for handle in hooks:
    handle.remove()
for index, layer_logits in enumerate(plain_logits):
    print(f"block {index}: {tok.decode(layer_logits[0, -1].argmax())!r}")
RESULT = plain_logits.shape == logits.shape

block 0: ' not'
block 1: ' now'
block 2: ' not'
block 3: ' not'
block 4: ' not'
block 5: ' not'
block 6: ' also'
block 7: ' Rome'
block 8: ' Rome'
block 9: ' Rome'
block 10: ' Rome'
block 11: ' Rome'


## The two agree

Every block's logits, at every position, are identical bit for bit.

In [4]:
# recipe-test: tool=transformers expected=True
assert torch.equal(logits, plain_logits)
print("identical:", tuple(logits.shape))
RESULT = torch.equal(logits, plain_logits)

identical: (12, 1, 12, 50257)
